2D Timeseries

predicting same feature as Input

Forecasting 1 timestep

Peter Resch, 29.5.

In [1]:
from __future__ import print_function, division   # Ensures Python3 printing & division standard
import pandas as pd 
from pandas import Series, DataFrame 
from matplotlib import pyplot as plt
import numpy as np

import seaborn as sns

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import sklearn
from sklearn.model_selection import train_test_split

import xarray as xr

rSeed=42

SavePlots = False

## Loading Data

In [14]:
filename="../common_tryout_datasets/small_grid/grid-timeseries_sel_vars_2026_april.nc"

# Open the NetCDF files using xarray
ds = xr.open_dataset(filename)
ds

<xarray.Dataset> Size: 136kB
Dimensions:  (time: 192, lat: 5, lon: 5)
Coordinates:
  * time     (time) datetime64[ns] 2kB 2026-04-09 ... 2026-04-16T23:00:00
  * lat      (lat) float64 40B 56.0 55.75 55.5 55.25 55.0
  * lon      (lon) float64 40B 12.0 12.25 12.5 12.75 13.0
Data variables:
    10u      (time, lat, lon) float32 19kB ...
    10v      (time, lat, lon) float32 19kB ...
    2t       (time, lat, lon) float32 19kB ...
    sp       (time, lat, lon) float32 19kB ...
    tp       (time, lat, lon) float32 19kB ...
    tcc      (time, lat, lon) float32 19kB ...
    ssrd     (time, lat, lon) float32 19kB ...
Attributes:
    CDI:          Climate Data Interface version 2.4.0 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    institution:  European Centre for Medium-Range Weather Forecasts
    history:      Fri May 29 15:23:50 2026: cdo -f nc copy grid-timeseries_se...
    CDO:          Climate Data Operators version 2.4.0 (https://mpimet.mpg.de...

In [131]:
class TimeseriesDatasetPipeline:
    """Pipeline to convert xarray Dataset to PyTorch Dataset with optional scaling."""
    
    def __init__(self, ds, scaler_set=None, forecast_lag=1, forecast_horizon=1,forecast_var=5):
        """
        Initialize the pipeline.
        
        Args:
            ds: xarray.Dataset with dimensions (time, lat, lon)
            scaler: sklearn scaler object (optional, e.g., StandardScaler)
            forecast_lag: number of past timesteps to use as input
            forecast_horizon: number of future timesteps to predict
        """
        self.scaler_set = [] if scaler_set is None else scaler_set
        self.ds = ds
        self.forecast_lag = forecast_lag
        self.forecast_horizon = forecast_horizon
        self.forecast_var = forecast_var
        self.data = None
        self.scaled_data = None
        self._prepare_data()
        if self.forecast_var >= self.data.shape[1]:
            raise ValueError(f"forecast_var index {self.forecast_var} is out of bounds for data with {self.data.shape[1]} variables.")
    
    def get_input_shape(self):
        """Return the shape of the input data (time, variables, space)."""
        return self.ds.to_array().values.shape if self.ds is not None else None

    def _prepare_data(self):
        """Convert xarray to numpy and flatten spatial dimensions."""
        # Stack lat and lon into single spatial dimension
        stacked = self.ds.to_array().stack(space=('lat', 'lon'))  #(variables, time, space)
        self.data = stacked.values.transpose(1, 0, 2)             #(time, variables, space)
        self._process_data()

    def _process_data(self):
        """Apply each scaler to the corresponding variable."""
        time, vars_, space = self.data.shape
        scaled = np.zeros_like(self.data)
        for i in range(vars_):
            reshaped = self.data[:, i, :].reshape(-1, 1)  # (time*space, 1)
            if len(self.scaler_set) <= i or self.scaler_set[i] is None:
                sc=sklearn.preprocessing.StandardScaler().fit(reshaped)
                self.scaler_set.append(sc)
            scaled[:, i, :] = self.scaler_set[i].transform(reshaped).reshape(time, space)
        self.scaled_data = scaled

    def set_scaler_set(self, scaler_set):
        """Set an array of scalers (one per variable) and reprocess data."""
        self.scaler_set = scaler_set
        self._process_data()

    def get_scaler_set(self):
        """Return the list/array of scalers."""
        return self.scaler_set

    def create_windowed_dataset(self):
        """Create X and y using moving window method."""
        data = self.scaled_data
        X, y = [], []
        for i in range(len(data) - self.forecast_lag - self.forecast_horizon + 1):
            #print(data.shape)
            data_filtered=data[i:i+self.forecast_lag,:,:]
            #print("data_filtered:", getattr(data_filtered, "shape", None), type(data_filtered))
            xd=np.delete(data_filtered, self.forecast_var, axis=1)# Remove target variable from input
            #print("x:", getattr(x, "shape", None), type(x),"\n")
            X.append(xd)
            yd=data[i+self.forecast_lag:i+self.forecast_lag+self.forecast_horizon,self.forecast_var,:]  # Only target variable
            y.append(yd)
        X = np.array(X)  # (samples, lag-1, space)
        y = np.array(y)  # (samples, horizon, space)
        print("X:", getattr(X, "shape", None),type(X))
        print("y:",getattr(y, "shape", None), type(y))
        return X,y
    
    def to_pytorch_dataset(self):
        """Convert to PyTorch Dataset."""
        X, y = self.create_windowed_dataset()
        return TimeseriesPyTorchDataset(X, y)

    def to_pytorch_dataloader(self, batch_size=32, shuffle=True):
        """Convert to PyTorch DataLoader."""
        dataset = self.to_pytorch_dataset()
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


class TimeseriesPyTorchDataset(Dataset):
    """PyTorch Dataset for timeseries data."""
    
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [132]:
pipeline = TimeseriesDatasetPipeline(ds, forecast_lag=24, forecast_horizon=1, forecast_var=5)
X,y=pipeline.create_windowed_dataset()


X: (168, 24, 6, 25) <class 'numpy.ndarray'>
y: (168, 1, 25) <class 'numpy.ndarray'>


In [133]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

validate_ratio=0.15
test_ratio=0.15
train_ratio=1-validate_ratio-test_ratio

n_total = len(ds.time)
n_test = int(n_total * test_ratio)
n_validate = int(n_total * validate_ratio)
n_train = n_total - n_test - n_validate

ds_test=ds.isel(time=slice(-n_test, None))  # Last n_test time steps for testing
ds_validate=ds.isel(time=slice(-n_test - n_validate, -n_test))  # Next n_validate time steps for validation
ds_train=ds.isel(time=slice(0, n_train))  # All but last n_test + n_validate for training

pipeline_train= TimeseriesDatasetPipeline(ds_train, forecast_lag=24, forecast_horizon=1)
dataloader_train = pipeline_train.to_pytorch_dataloader(batch_size=32, shuffle=True)
scaler_set = pipeline_train.get_scaler_set()

pipeline_validate = TimeseriesDatasetPipeline(ds_validate, scaler_set=scaler_set, forecast_lag=24, forecast_horizon=1)
dataloader_validate = pipeline_validate.to_pytorch_dataloader(batch_size=32, shuffle=True)

pipeline_test = TimeseriesDatasetPipeline(ds_test, scaler_set=scaler_set, forecast_lag=24, forecast_horizon=1)
dataloader_test = pipeline_test.to_pytorch_dataloader(batch_size=32, shuffle=False)

"""
for i in range(7):
    #plot histogram of first variable scaled and unscaled
    plt.figure(figsize=(6, 3))
    plt.subplot(1, 2, 1)
    plt.hist(pipeline.data[:, i, :].flatten(), bins=50, alpha=0.7, label='Unscaled')
    plt.title(f'Unscaled Variable {i+1}')
    plt.subplot(1, 2, 2)
    plt.hist(pipeline.scaled_data[:, i, :].flatten(), bins=50, alpha=0.7, label='Scaled')
    plt.title(f'Scaled Variable {i+1}')
    plt.tight_layout()
"""



X: (112, 24, 6, 25) <class 'numpy.ndarray'>
y: (112, 1, 25) <class 'numpy.ndarray'>
X: (4, 24, 6, 25) <class 'numpy.ndarray'>
y: (4, 1, 25) <class 'numpy.ndarray'>
X: (4, 24, 6, 25) <class 'numpy.ndarray'>
y: (4, 1, 25) <class 'numpy.ndarray'>


"\nfor i in range(7):\n    #plot histogram of first variable scaled and unscaled\n    plt.figure(figsize=(6, 3))\n    plt.subplot(1, 2, 1)\n    plt.hist(pipeline.data[:, i, :].flatten(), bins=50, alpha=0.7, label='Unscaled')\n    plt.title(f'Unscaled Variable {i+1}')\n    plt.subplot(1, 2, 2)\n    plt.hist(pipeline.scaled_data[:, i, :].flatten(), bins=50, alpha=0.7, label='Scaled')\n    plt.title(f'Scaled Variable {i+1}')\n    plt.tight_layout()\n"

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

hidden_size=16
raise NotImplementedError("The CNN-GRU model implementation is incomplete. Please implement the forward method and ensure the dimensions are correct before proceeding.")   
class CNN_GRU(nn.Module):
    def __init__(self,input_feature_size=6,hidden_size=16,lat_size=5,lon_size=5,output_size=1):
        super().__init__()

        def double_conv(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.ReLU(inplace=True)
            )
        #
        self.conv1 = double_conv(1, 16)
        self.maxpool = nn.MaxPool2d(kernel_size=(2, 2))
        self.conv2 = double_conv(16, 32)
        self.gru = nn.GRU(input_size=input_feature_size,hidden_size=hidden_size, num_layers=1,dropout=0,batch_first=True)
        self.linear = nn.Linear(hidden_size, output_size) # univariate output, so output size is 1
        

    def forward(self, x):
        print(x.shape)
        #recognize patterns of the spatial data with CNN
        x = self.conv1(x)
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)

        #print("in NeuralNetwork.forward")
        #print(x.shape)

        #decoding the temporal patterns with GRU
        x,y = self.gru(x)
        #print(x.shape,y.shape)
        x = x[:,-1,:].view(x.shape[0], 1,x.shape[2])#last hidden state
        #print(x.shape)
        x=self.linear(x)
        #print(x.shape)
        return x



cnn_gru_model=CNN_GRU().to(device)
print(cnn_gru_model)

# Loss function and optimizer
loss_fcn = nn.MSELoss()
optimizer_cnn_gru = AdamW(cnn_gru_model.parameters(), lr=1e-3, weight_decay=1e-4)

Using cpu device
CNN_GRU(
  (conv1): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
  )
  (maxpool): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (conv2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
  )
  (gru): GRU(6, 16, batch_first=True)
  (linear): Linear(in_features=16, out_features=6, bias=True)
)


In [134]:
def train(dataloader, model, loss_fn, optimizer,device):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        #print(X.shape, y.shape)
        X, y = X.to(device), y.to(device)
        #print(X.shape, y.shape)

        # Compute prediction error
        pred = model(X)#.squeeze()
        #print(pred)#.shape)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 10 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")



def test(dataloader, model, loss_fn,device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss = 0.0
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            predictions = model(X)
            test_loss += loss_fn(predictions, y).item()

            all_predictions.append(predictions.detach().cpu().numpy())
            all_targets.append(y.detach().cpu().numpy())

    test_loss /= num_batches
    y_pred = np.concatenate(all_predictions)#[:,:,0]
    y_true = np.concatenate(all_targets)#[:,:,0]
    print(f"y_true shape: {y_true.shape}, y_pred shape: {y_pred.shape}")

    mae = sklearn.metrics.mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(sklearn.metrics.mean_squared_error(y_true, y_pred))
    r2 = sklearn.metrics.r2_score(y_true, y_pred)
    print(f"Test Error:\n MAE: {mae:>8f}, RMSE: {rmse:>8f}, R2: {r2:>8f}, Avg loss: {test_loss:>8f} \n")
    return test_loss
